In [25]:
# Libraries 
import numpy as np 
import matplotlib.pyplot as plt 
import pandas as pd 
import librosa 
import soundfile as sf
from scipy.signal import butter, lfilter, freqz, correlate

In [26]:
# Load an audio file and adjust its sample rate to 20 kHz if required.

input_file = "I'm So Tired.wav"  # Specify the input audio file
signal, sample_rate = librosa.load(input_file, sr=None)  # Load the file and retrieve its sample rate

if sample_rate != 20000:  # Check if the sample rate differs from 20 kHz
    print(f"Resampling from {sample_rate} Hz to 20000 Hz...")  # Notify the resampling process
    signal = librosa.resample(signal, orig_sr=sample_rate, target_sr=20000)  # Resample to 20 kHz
    sample_rate = 20000  # Update the sample_rate variable for consistency

Resampling from 48000 Hz to 20000 Hz...


In [27]:
# Apply a second order low-pass filter to the signal
signal = lfilter(
    [0.24641218, 0.49282437, 0.24641218],  # Numerator coefficients for filtering
    [1., -0.75551434, 0.74116307],        # Denominator coefficients for filtering
    signal
)


In [28]:
# Decimate the signal through multiple stages and apply appropriate delays.

# Define decimation filter coefficients
filter_coefficients = [
    -0.0064, -0.0036, 0.0237, 0.0558, 0.0260, -0.1078, -0.2833, 0.3655,
    0.2833, 0.1078, -0.0260, -0.0558, -0.0237, 0.0036, 0.0064
]

# Define delay values (z^-x)
delays = {
    0: 49,  # Delay for the first stage (z^-49)
    1: 21,  # Delay for the second stage (z^-21)
    2: 7,   # Delay for the third stage (z^-7)
    3: 0    # No delay for the final stage (z^0)
}

# Placeholder for storing decimation products
decimation_products = []

# Process the signal through decimation stages
for stage in range(4):
    if stage > 0:  # Apply filtering and downsampling for stages 1 to 3
        filtered_signal = lfilter(filter_coefficients, [1], signal)  # Apply decimation filter
        signal = filtered_signal[::2]  # Downsample the signal by a factor of 2

    # Apply the appropriate delay to the signal
    delay = delays[stage]
    delayed_signal = np.pad(signal, (delay, 0), mode='constant')[:-delay] if delay > 0 else signal
    decimation_products.append(delayed_signal)  # Store the delayed signal

# Assign decimation products to variables
x_0, x_1, x_2, x_3 = decimation_products  # Decimated and delayed signals for further processing

In [29]:
# Apply bandpass filters to the decimation products and produce channel-specific outputs.

# Load the filters JSON into a DataFrame
filters_df = pd.read_json("bandpass filters.json")  # Load filter coefficients and settings

# Map decimation products to their respective identifiers
decimated_signals = {
    "x_0(n)": x_0,
    "x_1(n)": x_1,
    "x_2(n)": x_2,
    "x_3(n)": x_3
}

# Apply bandpass filters to each decimation product
channel_outputs = {}
for _, row in filters_df.iterrows():
    decimation_product = row["decimation_product"]  # Identify the decimation product to use
    signal = decimated_signals[decimation_product]  # Retrieve the corresponding signal
    b = row["filter_coefficients"]["numerator"]  # Numerator coefficients of the filter
    a = row["filter_coefficients"]["denominator"]  # Denominator coefficients of the filter
    channel_outputs[f"channel_{row['band_index']}"] = lfilter(b, a, signal)  # Filter the signal


In [30]:
# Simulate hair cell responses by applying rectification, compression, and smoothing.

# Function to apply the hair cell model
def hair_cell_model(signal, gamma=0.4, cutoff_freq=300, fs=20000):
    rectified = np.maximum(signal, 0)  # Perform half-wave rectification
    compressed = np.power(rectified, gamma)  # Apply non-linear compression
    b, a = butter(2, cutoff_freq / (fs / 2), btype='low')  # Design low-pass Butterworth filter
    smoothed = lfilter(b, a, compressed)  # Smooth the signal using low-pass filtering
    return smoothed

# Process each channel through the hair cell model
hair_cell_outputs = {}
for channel, signal in channel_outputs.items():
    hair_cell_outputs[channel] = hair_cell_model(signal)  # Apply HCM to each channel output

In [31]:
# Function to simulate auditory nerve response with low-pass filtering

def auditory_nerve_lowpass(signal, cutoff_freq=300, fs=20000):
    b, a = butter(2, cutoff_freq / (fs / 2), btype='low')  # Design low-pass filter
    return lfilter(b, a, signal)  # Filter the signal

# Process each channel through the auditory nerve model
auditory_nerve_outputs = {
    channel: auditory_nerve_lowpass(signal)  # Apply low-pass filtering to each HCM output
    for channel, signal in hair_cell_outputs.items()
}



In [32]:
# Analyze periodicity and tonal features using autocorrelation and feature summation.

# Function to compute autocorrelation for periodicity detection
def autocorrelate(signal, fs, max_lag_ms=50):
    max_lag_samples = int((max_lag_ms / 1000) * fs)  # Compute maximum lag in samples
    autocorr = correlate(signal, signal, mode='full')  # Perform full autocorrelation
    mid = len(autocorr) // 2  # Identify the central point
    return autocorr[mid : mid + max_lag_samples]  # Return the autocorrelation result for relevant lags

# Compute autocorrelation for each channel and store the results
cpu_outputs = {}
for channel, signal in auditory_nerve_outputs.items():
    cpu_outputs[channel] = autocorrelate(signal, fs=20000)  # Perform periodicity detection

# Combine outputs from all channels to analyze tonal/rhythmic features
combined_features = sum(cpu_outputs.values())  # Summation of periodicity results across channels